In [1]:
import time
from pprint import pprint
from IPython.display import display

from fda_ode import *

In [2]:
eps, t, h = symbols(r'\varepsilon, t, h', real=True)
u, u1 = symbols('u, u1', cls=Function)

In [3]:
eq = u(t).diff(t, 2)  + u(t) + eps*u(t)**3
eq

\varepsilon*u(t)**3 + u(t) + Derivative(u(t), (t, 2))

## Nayfeh A. H. Introduction to Perturbation Techniques, Wiley-VCH, 1st edition, 1993. 536 p.

In [4]:
eq1 = (u(t).diff(t)**2/2 + u(t)**2/2 + eps*u(t)**4/4) - h

In [5]:
expand(eq1.diff(t) - eq*u.diff(t))

\varepsilon*u(t)**3*Derivative(u(t), t) - \varepsilon*u(t)**3 + u(t)*Derivative(u(t), t) - u(t) + Derivative(u(t), t)*Derivative(u(t), (t, 2)) - Derivative(u(t), (t, 2))

In [6]:
init((u, u1), t, h)

In [7]:
def f(y):
    return Matrix([\
        y[1],\
        -y[0] - eps*y[0]**3,\
    ])

s = (('u(t)', 'y[0]'), ('u1(t)', 'y[1]'), (r'\varepsilon', 'eps'))
def replace(r):
    for a, b in s:
        r = r.replace(a, b)
    return r

In [8]:
start = time.time()

## RadauIIA 1 order

In [9]:
F = RadauIIA1(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-u1_{t}


h^0 =>


\varepsilon*u**3 + u + u1_{t}

h^1 =>


3*\varepsilon*u**2*u_{t} + u_{t}

In [10]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


\varepsilon*u**3 + u


h^0 =>


\varepsilon*u**3 + u + u1_{t}

h^1 =>


3*\varepsilon*u**2*u1 + u1

In [11]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


-\varepsilon**2*u**6/2 - \varepsilon*u**2*(2*u**2 + 3*u1**2)/2 - (u**2 + u1**2)/2

In [12]:
print(replace(str(fda[1][1].collect(eps))))

3*eps*y[0]**2*y[1] + y[1]


## Gauss–Legendre

In [13]:
F = GaussLegendre2(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-(u1_{t} - u_{tt})/2

h^2 =>


...

\varepsilon*u**3 + u + u1_{t}

h^1 =>


3*\varepsilon*u**2*u_{t}/2 + (u1_{tt} + u_{t})/2

h^2 =>


3*\varepsilon*u*(u*u_{tt} + u_{t}**2)/4 + u_{tt}/4

In [14]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

\varepsilon*u**3 + u + u1_{t}

h^1 =>


0

h^2 =>


-3*\varepsilon**2*u**5/4 - \varepsilon*u*(4*u**2 - 3*u1**2)/4 - u/4

In [15]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


\varepsilon*u*u1**3/4

In [16]:
print(replace(str(fda[1][2].collect(eps))))

-3*eps**2*y[0]**5/4 + eps*(-y[0]**3 + 3*y[0]*y[1]**2/4) - y[0]/4


## Crank–Nicolson

In [17]:
F = CrankNicolson2(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-(u1_{t} - u_{tt})/2

h^2 =>


...

\varepsilon*u**3 + u + u1_{t}

h^1 =>


3*\varepsilon*u**2*u_{t}/2 + (u1_{tt} + u_{t})/2

h^2 =>


3*\varepsilon*u*(u*u_{tt} + 2*u_{t}**2)/4 + u_{tt}/4

In [18]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

\varepsilon*u**3 + u + u1_{t}

h^1 =>


0

h^2 =>


-3*\varepsilon**2*u**5/4 - \varepsilon*u*(2*u**2 - 3*u1**2)/2 - u/4

In [19]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


-\varepsilon*u*u1**3/2

In [20]:
print(replace(str(fda[1][2].collect(eps))))

-3*eps**2*y[0]**5/4 + eps*(-y[0]**3 + 3*y[0]*y[1]**2/2) - y[0]/4


## RadauIIA 3 order

In [21]:
F = RadauIIA3(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-\varepsilon*u**3/6 - (u + 4*u1_{t} - 3*u_{tt})/6

h^2 =>


...

-\varepsilon*u**2*(u1 - 4*u_{t})/2 - (u1 - 3*u1_{tt} - 4*u_{t})/6

h^2 =>


-\varepsilon*u*(9*u*u1_{t} - 18*u*u_{tt} - 2*u1**2 + 10*u1*u_{t} - 26*u_{t}**2)/18 + (u1_{ttt} - u1_{t} + 2*u_{tt})/6

h^3 =>


-\varepsilon*(243*u**2*u1_{tt} - 324*u**2*u_{ttt} - 216*u*u1*u1_{t} + 270*u*u1*u_{tt} + 540*u*u1_{t}*u_{t} - 1404*u*u_{tt}*u_{t} + 8*u1**3 - 60*u1**2*u_{t} + 150*u1*u_{t}**2 - 368*u_{t}**3)/972 - (3*u1_{tt} - 4*u_{ttt})/36

In [22]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^2 =>


0

h^3 =>


-13*\varepsilon**2*u**4*u1/12 - \varepsilon*u1*(18*u**2 - 5*u1**2)/18 - u1/36

In [23]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


0

h^3 =>


-\varepsilon**3*u**8/24 - \varepsilon**2*u**4*(7*u**2 - 3*u1**2)/72 - \varepsilon*(5*u**4 - 6*u**2*u1**2 + 2*u1**4)/72 - (u**2 + u1**2)/72

In [24]:
print(replace(str(fda[1][3].collect(eps))))

-13*eps**2*y[0]**4*y[1]/12 + eps*(-y[0]**2*y[1] + 5*y[1]**3/18) - y[1]/36


## "original" Runge–Kutta

In [25]:
F = RungeKutta4(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


\varepsilon*u**3/2 + (u + u_{tt})/2

h^2 =>


...

-\varepsilon**2*u**5/2 - \varepsilon*u*(2*u**2 - 3*u1**2)/3 - (u - u1_{ttt})/6

h^3 =>


-9*\varepsilon**2*u**4*u1/8 - \varepsilon*u1*(2*u - u1)*(2*u + u1)/4 - (u1 - u1_{tttt})/24

h^4 =>


3*\varepsilon**3*u**7/16 + \varepsilon**2*u**3*(3*u**2 - 10*u1**2)/8 + \varepsilon*u*(3*u**2 - 10*u1**2)/16

In [26]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^3 =>


0

h^4 =>


3*\varepsilon**3*u**7/16 + \varepsilon**2*u**3*(3*u**2 - 10*u1**2)/8 + \varepsilon*u*(3*u**2 - 10*u1**2)/16

In [27]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


0

h^3 =>


0

h^4 =>


-\varepsilon**3*u**7*u1/16 - \varepsilon**2*u**3*u1*(3*u**2 - 5*u1**2)/24 - \varepsilon*u*u1*(3*u**2 - 4*u1**2)/48

In [28]:
print(replace(str(fda[1][4].collect(eps))))

3*eps**3*y[0]**7/16 + eps**2*(3*y[0]**5/8 - 5*y[0]**3*y[1]**2/4) + eps*(3*y[0]**3/16 - 5*y[0]*y[1]**2/8)


## Dormand–Prince 5 order

In [29]:
F = DormandPrince5(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


\varepsilon*u**3/2 + (u + u_{tt})/2

h^2 =>


...

-9*\varepsilon**2*u**4*u1/8 - \varepsilon*u1*(2*u - u1)*(2*u + u1)/4 - (u1 - u1_{tttt})/24

h^4 =>


9*\varepsilon**3*u**7/40 + \varepsilon**2*u**3*(17*u**2 - 42*u1**2)/40 + \varepsilon*u*(25*u**2 - 66*u1**2)/120 + (u + u1_{ttttt})/120

h^5 =>


4427*\varepsilon**3*u**6*u1/7200 + \varepsilon**2*u**2*u1*(3197*u**2 - 1890*u1**2)/3600 + \varepsilon*u1*(415*u**2 - 132*u1**2)/1440 + u1/600

In [30]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


4427*\varepsilon**3*u**6*u1/7200 + \varepsilon**2*u**2*u1*(3197*u**2 - 1890*u1**2)/3600 + \varepsilon*u1*(415*u**2 - 132*u1**2)/1440 + u1/600

In [31]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


-19*\varepsilon**4*u**10/7200 - \varepsilon**3*u**6*(17*u**2 + 5*u1**2)/2400 - 47*\varepsilon**2*u**6/7200 - \varepsilon*u**2*(17*u**2 + 3*u1**2)/7200 - (u**2 + u1**2)/3600

In [32]:
print(replace(str(fda[1][5].collect(eps))))

0


### Adams–Bashforth 4 order

In [33]:
F = AdamsBashforth4(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-7*(u1_{t} - u_{tt})/2

h^2 =>


...

37*\varepsilon*u*(u*u_{tt} + 2*u_{t}**2)/2 + 37*(u1_{ttt} + u_{tt})/6

h^3 =>


175*\varepsilon*(u**2*u_{ttt} + 6*u*u_{tt}*u_{t} + 2*u_{t}**3)/8 + 175*(u1_{tttt} + u_{ttt})/24

h^4 =>


887*\varepsilon*(u**2*u_{tttt} + 8*u*u_{ttt}*u_{t} + 6*u*u_{tt}**2 + 12*u_{tt}*u_{t}**2)/48 + 887*u_{tttt}/144

In [34]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^3 =>


0

h^4 =>


2661*\varepsilon**3*u**7/16 + 887*\varepsilon**2*u**3*(17*u**2 - 42*u1**2)/48 + 887*\varepsilon*u*(25*u**2 - 66*u1**2)/144 + 887*u/144

In [35]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


0

h^3 =>


0

h^4 =>


4429*\varepsilon**2*u**3*u1**3/6 + 4429*\varepsilon*u*u1**3/12

In [36]:
print(replace(str(fda[1][4].collect(eps))))

2661*eps**3*y[0]**7/16 + eps**2*(15079*y[0]**5/48 - 6209*y[0]**3*y[1]**2/8) + eps*(22175*y[0]**3/144 - 9757*y[0]*y[1]**2/24) + 887*y[0]/144


### Adams–Bashforth 5 order

In [37]:
F = AdamsBashforth5(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-9*(u1_{t} - u_{tt})/2

h^2 =>


...

369*\varepsilon*(u**2*u_{ttt} + 6*u*u_{tt}*u_{t} + 2*u_{t}**3)/8 + 123*(u1_{tttt} + u_{ttt})/8

h^4 =>


2101*\varepsilon*(u**2*u_{tttt} + 8*u*u_{ttt}*u_{t} + 6*u*u_{tt}**2 + 12*u_{tt}*u_{t}**2)/40 + 2101*(u1_{ttttt} + u_{tttt})/120

h^5 =>


22583*\varepsilon*(u**2*u_{ttttt} + 10*u*u_{tttt}*u_{t} + 20*u*u_{ttt}*u_{tt} + 20*u_{ttt}*u_{t}**2 + 30*u_{tt}**2*u_{t})/480 + 22583*u_{ttttt}/1440

In [38]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


1106567*\varepsilon**3*u**6*u1/160 + 22583*\varepsilon**2*u**2*u1*(71*u**2 - 42*u1**2)/160 + 22583*\varepsilon*u1*(69*u**2 - 22*u1**2)/480 + 22583*u1/1440

In [39]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


-67743*\varepsilon**4*u**10/160 - 7527*\varepsilon**3*u**6*(26*u**2 + 105*u1**2)/160 - 2509*\varepsilon**2*u**2*(4*u**2 - 3*u1**2)*(19*u**2 + 126*u1**2)/160 - 2509*\varepsilon*(26*u**4 + 141*u**2*u1**2 - 66*u1**4)/160 - 2509*(u**2 + u1**2)/160

In [40]:
print(replace(str(fda[1][5].collect(eps))))

1106567*eps**3*y[0]**6*y[1]/160 + eps**2*(1603393*y[0]**4*y[1]/160 - 474243*y[0]**2*y[1]**3/80) + eps*(519409*y[0]**2*y[1]/160 - 248413*y[1]**3/240) + 22583*y[1]/1440


### Adams–Moulton 4 order

In [41]:
F = AdamsMoulton4(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-7*(u1_{t} - u_{tt})/2

h^2 =>


...

37*\varepsilon*u*(u*u_{tt} + 2*u_{t}**2)/2 + 37*(u1_{ttt} + u_{tt})/6

h^3 =>


175*\varepsilon*(u**2*u_{ttt} + 6*u*u_{tt}*u_{t} + 2*u_{t}**3)/8 + 175*(u1_{tttt} + u_{ttt})/24

h^4 =>


781*\varepsilon*(u**2*u_{tttt} + 8*u*u_{ttt}*u_{t} + 6*u*u_{tt}**2 + 12*u_{tt}*u_{t}**2)/40 + 781*u_{tttt}/120

In [42]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^3 =>


0

h^4 =>


7029*\varepsilon**3*u**7/40 + 781*\varepsilon**2*u**3*(17*u**2 - 42*u1**2)/40 + 781*\varepsilon*u*(25*u**2 - 66*u1**2)/120 + 781*u/120

In [43]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


0

h^3 =>


0

h^4 =>


780*\varepsilon**2*u**3*u1**3 + 390*\varepsilon*u*u1**3

In [44]:
print(replace(str(fda[1][4].collect(eps))))

7029*eps**3*y[0]**7/40 + eps**2*(13277*y[0]**5/40 - 16401*y[0]**3*y[1]**2/20) + eps*(3905*y[0]**3/24 - 8591*y[0]*y[1]**2/20) + 781*y[0]/120


### Adams–Moulton 5 order

In [45]:
F = AdamsMoulton5(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


-9*(u1_{t} - u_{tt})/2

h^2 =>


...

369*\varepsilon*(u**2*u_{ttt} + 6*u*u_{tt}*u_{t} + 2*u_{t}**3)/8 + 123*(u1_{tttt} + u_{ttt})/8

h^4 =>


2101*\varepsilon*(u**2*u_{tttt} + 8*u*u_{ttt}*u_{t} + 6*u*u_{tt}**2 + 12*u_{tt}*u_{t}**2)/40 + 2101*(u1_{ttttt} + u_{tttt})/120

h^5 =>


3843*\varepsilon*(u**2*u_{ttttt} + 10*u*u_{tttt}*u_{t} + 20*u*u_{ttt}*u_{tt} + 20*u_{ttt}*u_{t}**2 + 30*u_{tt}**2*u_{t})/80 + 1281*u_{ttttt}/80

In [46]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    print()

h^0 =>


-u1 + u_{t}

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


564921*\varepsilon**3*u**6*u1/80 + 11529*\varepsilon**2*u**2*u1*(71*u**2 - 42*u1**2)/80 + 3843*\varepsilon*u1*(69*u**2 - 22*u1**2)/80 + 1281*u1/80

In [47]:
F3 = ((T(u1(t+h))**2/2 + T(u(t+h))**2/2 + eps*T(u(t+h))**4/4) -\
      (T(u1(t))**2/2 + T(u(t))**2/2 + eps*T(u(t))**4/4))/h
f3 = fda.NF(F3)
f3.prn(eps)

h^0 =>


0

h^1 =>


0

h^2 =>


...

0

h^4 =>


0

h^5 =>


-4323*\varepsilon**4*u**10/10 - 1441*\varepsilon**3*u**6*(26*u**2 + 105*u1**2)/30 - 1441*\varepsilon**2*u**2*(4*u**2 - 3*u1**2)*(19*u**2 + 126*u1**2)/90 - 1441*\varepsilon*(26*u**4 + 141*u**2*u1**2 - 66*u1**4)/90 - 1441*(u**2 + u1**2)/90

In [48]:
print(replace(str(fda[1][5].collect(eps))))

564921*eps**3*y[0]**6*y[1]/80 + eps**2*(818559*y[0]**4*y[1]/80 - 242109*y[0]**2*y[1]**3/40) + eps*(265167*y[0]**2*y[1]/80 - 42273*y[1]**3/40) + 1281*y[1]/80


In [49]:
end = time.time()
print(end - start)

35.207130432128906
